In [1]:
from pathlib import Path

import pandas as pd
from pandas.testing import assert_frame_equal

In [2]:
PROJECT_ROOT = Path.cwd().resolve().parent

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"

In [3]:
FILE_2026_08 = RAW_DIR / "LIMA_METROPOLITANA_2026_08.xlsx"
FILE_2026_07 = RAW_DIR / "LIMA_METROPOLITANA_2026_07.xlsx"
FILE_2026_06 = RAW_DIR / "LIMA_METROPOLITANA_2026_06.xlsx"
FILE_2026_05 = RAW_DIR / "LIMA_METROPOLITANA_2026_05.xlsx"

FILE_2025_08 = RAW_DIR / "LIMA_METROPOLITANA_2025_08.xlsx"

In [4]:
PATH_2025_08 = INTERIM_DIR / "pt_2025_08.parquet"
PATH_2026_08 = INTERIM_DIR / "pt_2026_08.parquet"

In [5]:
df_2026_08 = pd.read_excel(
    FILE_2026_08,
    sheet_name="LIMA_METRPOLITANA_2026_08",
    header=3
)

In [6]:
df_2026_08.head(1000).info(
    verbose=True,
    show_counts=True
)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 141 columns):
 #    Column      Non-Null Count  Dtype  
---   ------      --------------  -----  
 0    codigo_eje  1000 non-null   int64  
 1    nombre_eje  1000 non-null   object 
 2    coddisa     1000 non-null   int64  
 3    nomdisa     1000 non-null   object 
 4    codred      977 non-null    float64
 5    red         977 non-null    object 
 6    codmef      977 non-null    float64
 7    uemef       974 non-null    object 
 8    codigo_pre  1000 non-null   object 
 9    establec    1000 non-null   object 
 10   tipo        1000 non-null   object 
 11   CATEGORIA   935 non-null    object 
 12   est_act     1000 non-null   object 
 13   ue          157 non-null    object 
 14   quintil     1000 non-null   int64  
 15   vraem       898 non-null    object 
 16   europ       898 non-null    object 
 17   indig       898 non-null    object 
 18   codigo_med  1000 non-null   int64  
 19   nombr

In [7]:
if not PATH_2026_08.exists():
    df_2026_08.to_parquet(
        PATH_2026_08,
        engine="pyarrow",
        index=False,
    )

In [8]:
pt_2026_08 = pd.read_parquet(
    INTERIM_DIR / "pt_2026_08.parquet",
    engine="pyarrow"
)

In [9]:
pt_2026_08.head(1000).info(
    verbose=True,
    show_counts=True
)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 141 columns):
 #    Column      Non-Null Count  Dtype         
---   ------      --------------  -----         
 0    codigo_eje  1000 non-null   int64         
 1    nombre_eje  1000 non-null   object        
 2    coddisa     1000 non-null   int64         
 3    nomdisa     1000 non-null   object        
 4    codred      977 non-null    float64       
 5    red         977 non-null    object        
 6    codmef      977 non-null    float64       
 7    uemef       974 non-null    object        
 8    codigo_pre  1000 non-null   object        
 9    establec    1000 non-null   object        
 10   tipo        1000 non-null   object        
 11   CATEGORIA   935 non-null    object        
 12   est_act     1000 non-null   object        
 13   ue          157 non-null    object        
 14   quintil     1000 non-null   int64         
 15   vraem       898 non-null    object        
 16   europ

In [10]:
try:
    assert_frame_equal(
        df_2026_08,
        pt_2026_08,
        check_dtype=True,
        check_exact=True
    )
except AssertionError as e:
    print(f"Assertion error: {e}")

Assertion error: Attributes of DataFrame.iloc[:, 128] (column name="fecha_venc") are different

Attribute "dtype" are different
[left]:  object
[right]: datetime64[us]


In [11]:
df_2026_08["codigo_pre"].map(type).value_counts()

codigo_pre
<class 'str'>    206674
Name: count, dtype: int64

In [12]:
df_2026_08["fecha_venc"].map(type)

0         <class 'datetime.datetime'>
1         <class 'datetime.datetime'>
2                     <class 'float'>
3                     <class 'float'>
4         <class 'datetime.datetime'>
                     ...             
206669    <class 'datetime.datetime'>
206670    <class 'datetime.datetime'>
206671    <class 'datetime.datetime'>
206672    <class 'datetime.datetime'>
206673    <class 'datetime.datetime'>
Name: fecha_venc, Length: 206674, dtype: object

In [13]:
df_2026_08["fecha_venc"].map(type).value_counts()

fecha_venc
<class 'datetime.datetime'>    173158
<class 'float'>                 33516
Name: count, dtype: int64

In [14]:
# df_2026_08["fecha_venc"][df_2026_08["fecha_venc"].map(type) == 'float64'].head(10)

In [153]:
df_2026_08.loc[
    df_2026_08["fecha_venc"].map(type) == float,
    "fecha_venc"
].value_counts(dropna=False)

fecha_venc
NaN    33516
Name: count, dtype: int64

In [154]:
df_2026_08["fecha_venc"] = df_2026_08["fecha_venc"].astype(
    "datetime64[us]"
)

In [155]:
assert_frame_equal(
    df_2026_08,
    pt_2026_08,
    check_dtype=True,
    check_exact=True,
)

In [18]:
df_2026_08["codigo_pre"] = df_2026_08["codigo_pre"].astype("string")
df_2026_08["codigo_pre"].isna().sum()

np.int64(0)

In [19]:
# df_2026_08["fecha_venc"] = pd.to_datetime(df_2026_08["fecha_venc"], errors="raise")
# Auditar

In [20]:
print("Dimensiones (2026-08):", df_2026_08.shape)
print("Dimensiones (parquet):", pt_2026_08.shape)

Dimensiones (2026-08): (206674, 141)
Dimensiones (parquet): (206674, 141)


In [21]:
print("Coincidencia de DataFrame:", df_2026_08.shape == pt_2026_08.shape)

Coincidencia de DataFrame: True


In [22]:
print("Tipo Parquet:", pt_2026_08["fecha_venc"].dtype)

Tipo Parquet: datetime64[us]


In [23]:
pt_2026_08["fecha_venc"].map(type).value_counts()

fecha_venc
<class 'pandas._libs.tslibs.timestamps.Timestamp'>    173158
<class 'pandas._libs.tslibs.nattype.NaTType'>          33516
Name: count, dtype: int64

In [24]:
print("Menor fecha:", pt_2026_08["fecha_venc"].min())
print("Mayor fecha:", pt_2026_08["fecha_venc"].max())

print("Cantidad de NaT:", pt_2026_08["fecha_venc"].isna().sum())

Menor fecha: 1900-01-01 00:00:00
Mayor fecha: 3031-06-30 00:00:00
Cantidad de NaT: 33516


In [156]:
# Parte 2:

def get_categorical_columns(df):
    return df.select_dtypes(
        include=['object', 'category', 'string']
    ).columns.tolist()

In [157]:
categorical_columns = get_categorical_columns(pt_2026_08)
categorical_columns

['nombre_eje',
 'nomdisa',
 'red',
 'uemef',
 'codigo_pre',
 'establec',
 'tipo',
 'CATEGORIA',
 'est_act',
 'ue',
 'vraem',
 'europ',
 'indig',
 'nombre_med',
 'formaf',
 'tipomed',
 'subtipo',
 'Nomsubtipo',
 'petitorio',
 'estrategic',
 'tipsum',
 'indicador']

In [20]:
def get_value_counts(df, column):
    return df[column].value_counts(dropna=False)

In [28]:
get_value_counts(pt_2026_08, "vraem")

vraem
        186843
None     19831
Name: count, dtype: int64

In [29]:
get_value_counts(pt_2026_08, "europ")

europ
        186843
None     19831
Name: count, dtype: int64

In [159]:
get_value_counts(pt_2026_08, "indig")

indig
        186843
None     19831
Name: count, dtype: int64

In [31]:
get_value_counts(pt_2026_08, "nombre_eje")

nombre_eje
LIMA METROPOLITANA    206674
Name: count, dtype: int64

In [32]:
get_value_counts(pt_2026_08, "nomdisa")

nomdisa
LIMA CENTRO    58608
LIMA SUR       55524
LIMA NORTE     52327
LIMA ESTE      40215
Name: count, dtype: int64

In [160]:
get_value_counts(pt_2026_08, "red")

red
NO PERTENECE A NINGUNA RED                          186711
LIMA CIUDAD                                           6122
None                                                  4864
LIMA ESTE METROPOLITANA                               3202
SAN JUAN DE LURIGANCHO                                2130
TUPAC AMARU                                           1347
V  RIMAC-SMP-LO                                       1273
RED DE SALUD LIMA NORTE IV                             666
SAN JUAN DE MIRAFLORES - VILLA MARIA DEL TRIUNFO       335
                                                        24
Name: count, dtype: int64

In [34]:
get_value_counts(pt_2026_08, "uemef") # C

uemef
DIRECCION DE REDES INTEGRADAS DE SALUD LIMA SUR         50493
DIRECCION DE REDES INTEGRADAS DE SALUD LIMA NORTE       47381
DIRECCION DE REDES INTEGRADAS DE SALUD LIMA CENTRO      37718
DIRECCION DE REDES INTEGRADAS DE SALUD LIMA ESTE        33172
None                                                     5447
HOSPITAL NACIONAL ARZOBISPO LOAYZA                       2253
INSTITUTO NACIONAL DE SALUD DEL NIÑO SAN BORJA           2244
HOSPITAL NACIONAL HIPOLITO UNANUE                        1962
HOSPITAL NACIONAL DOS DE MAYO                            1893
HOSPITAL DE APOYO DEPARTAMENTAL MARIA AUXILIADORA        1841
HOSPITAL CAYETANO HEREDIA                                1755
HOSPITAL DE EMERGENCIAS VILLA EL SALVADOR                1750
INSTITUTO NACIONAL DE SALUD DEL NIÑO                     1737
HOSPITAL SAN JUAN DE LURIGANCHO                          1655
HOSPITAL DE APOYO SANTA ROSA                             1575
INSTITUTO NACIONAL DE ENFERMEDADES NEOPLASICAS           1425
HO

In [161]:
get_value_counts(pt_2026_08, "codigo_pre")

codigo_pre
06207    2253
16918    2244
05946    1962
06206    1893
05987    1841
         ... 
23553       8
10385       7
24546       7
20382       6
08706       6
Name: count, Length: 509, dtype: int64

In [36]:
get_value_counts(pt_2026_08, "establec")

establec
HOSPITAL NACIONAL ARZOBISPO LOAYZA                                        2253
INSTITUTO NACIONAL DE SALUD NIÑO SAN BORJA                                2244
HOSPITAL NACIONAL HIPOLITO UNANUE                                         1962
HOSPITAL NACIONAL DOS DE MAYO                                             1893
HOSPITAL MARIA AUXILIADORA                                                1841
                                                                          ... 
POLICLINICO MILITAR CHORRILLOS SANIDAD DEL EJERCITO DEL PERU                 8
POSTA FAP SAN GABINO SANIDAD DE LA FUERZA AEREA DEL PERU                     7
CENTRO DE SALUD MILITAR "SAN JUAN" SANIDAD DEL EJERCITO DEL PERU             7
POLICLINICO PNP CHORRILLOS SANIDAD DE LA POLICIA NACIONAL DEL PERU           6
ESTABLECIMIENTO DE SALUD MILITAR - COEDE SANIDAD DEL EJERCITO DEL PERU       6
Name: count, Length: 494, dtype: int64

In [162]:
get_value_counts(pt_2026_08, "tipo")

tipo
C.S.          111583
P.S.           50319
HOSPITAL       23330
INSTITUTO       8055
ALMACEN         7028
FARMACIA D      4372
OTROS           1554
SUBALM           433
Name: count, dtype: int64

In [38]:
get_value_counts(pt_2026_08, "CATEGORIA")

CATEGORIA
I-3      89939
I-2      50542
I-4      21642
III-1    14089
None     11833
III-2     8055
III-E     4663
II-2      3410
II-1      1234
SIN C      995
I-1        272
Name: count, dtype: int64

In [163]:
get_value_counts(pt_2026_08, "est_act")

est_act
A    204123
I      2551
Name: count, dtype: int64

In [164]:
get_value_counts(pt_2026_08, "ue")

ue
None    175289
U        31385
Name: count, dtype: int64

In [41]:
get_value_counts(pt_2026_08, "nombre_med")

nombre_med
MASCARILLA DESCARTABLE QUIRURGICA 3 PLIEGUES   UNIDAD                          553
BEDAQUILINA 100 mg  TABLETA                                                    549
LINEZOLID 600 mg  TABLETA                                                      549
ALCOHOL ETILICO (ETANOL) 70° 1 L SOLUCION                                      490
JERINGA DESCARTABLE 5 mL CON AGUJA 21 G X 1 1/2"   UNIDAD                      487
                                                                              ... 
HORMONA TRIYODOTIRONINA (T3) TOTAL  100 DET UNIDAD                               1
SET COLORANTE GRAM DE 500 mL  (5 FRASCOS) UNIDAD                                 1
TIRA DE LIJA DE 4 mm PARA RESINA X 150   UNIDAD                                  1
FRASCO DE VIDRIO COLOR AMBAR CON TAPA Y SELLO DE SEGURIDAD X 60 mL   UNIDAD      1
INTRODUCTOR VASCULAR 5 FR X 11 cm   UNIDAD                                       1
Name: count, Length: 10472, dtype: int64

In [165]:
pt_2026_08["nombre_med"].nunique()

10471

In [43]:
get_value_counts(pt_2026_08, "formaf")

formaf
UNIDAD                              87968
TABLETA                             42098
INYECTABLE                          32660
KIT                                 10393
SOLUCION                            10298
SUSPENSION                           8980
CREMA                                3368
JARABE                               2624
AEROSOL                              1941
PAR                                  1662
GEL                                  1256
POLVO                                1189
UNGUENTO                              600
LOCION                                412
OVULO                                 406
GAS                                   399
SET                                   171
ELIXIR                                101
JABON LIQUIDO                          87
POMADA                                 17
PARCHE                                 14
GRANULOS                               14
SUPOSITORIO                             5
None                       

In [44]:
get_value_counts(pt_2026_08, "tipomed")

tipomed
M       103503
I       103170
None         1
Name: count, dtype: int64

In [166]:
get_value_counts(pt_2026_08, "subtipo")

subtipo
M               88094
INSUMOS         65147
R               14741
POR DEFINIR     14649
PB               9300
IM               3590
PG               3382
PS               2110
PD               1438
AYDE             1389
O                 862
IO                720
MEDICAMENTO       505
AD                314
NP                189
I                  68
A                  54
MEDICAMENTOS       49
ADX                25
MQ                 19
L                  17
NE                  7
PN                  3
None                1
PR                  1
Name: count, dtype: int64

In [32]:
dicc_pairs = {
    "MEDICAMENTO": "M",
    "INSUMOS": "I",
    "REACTIVOS Y MATERIAL DE LABORATORIO": "R",
    "POR DEFINIR": "POR DEFINIR",
    "PRODUCTO BIOLÓGICO": "PB",
    "INSTRUMENTAL MEDICO": "IM",
    "PRODUCTO GALENICO": "PG",
    "PRODUCTO SANITARIO": "PS",
    "PRODUCTO DIETÉTICO": "PD",
    "ANTISEPTICO Y DESINFECTANTE": "AYDE",
    "OTROS": "O",
    "INTRUMENTAL ODONTOLOGICO": "IO",
    "NUTRICION PARENTERAL": "NP",
    "ALIMENTOS, CEREALES, FORMULAS Y NUTRIENTES": "A",
    "AGENTE DE DIAGNOSTICO": "AD",
    "MEDICAMENTOS QUIMICOS": "MQ",
    "SUCEDANEOS DE LA LECHE MATERNA": "L",
    "NUTRICION ENTERAL": "NE",
    "NO_DIGESA": "ND",
    "PRODUCTO NATURAL": "PN",
    "PRODUCTO RADIOFARMACO": "PR"
}

In [47]:
get_value_counts(pt_2026_08, "Nomsubtipo")

Nomsubtipo
MEDICAMENTO                                   88598
INSUMOS                                       65204
REACTIVOS Y MATERIAL DE LABORATORIO           14741
POR DEFINIR                                   14649
PRODUCTO BIOLÓGICO                             9300
INSTRUMENTAL MEDICO                            3590
PRODUCTO GALENICO                              2778
PRODUCTO SANITARIO                             2110
PRODUCTO DIETÉTICO                             1438
ANTISEPTICO Y DESINFECTANTE                    1389
OTROS                                           862
INTRUMENTAL ODONTOLOGICO                        720
GALENICOS                                       604
ANTISEPTICOS Y DESINFECTANTES                   314
NUTRICION PARENTERAL                            189
ALIMENTOS, CEREALES, FORMULAS Y NUTRIENTES       54
MEDICAMENTOS                                     49
AGENTE DE DIAGNOSTICO                            25
MEDICAMENTOS QUIMICOS                            19
S

In [48]:
get_value_counts(pt_2026_08, "petitorio")

petitorio
P       142643
_        64030
None         1
Name: count, dtype: int64

In [49]:
get_value_counts(pt_2026_08, "estrategic")

estrategic
SD       93244
SC/SD    87331
SC       26098
None         1
Name: count, dtype: int64

In [50]:
get_value_counts(pt_2026_08, "tipsum")

tipsum
S    202798
D      3876
Name: count, dtype: int64

In [51]:
get_value_counts(pt_2026_08, "indicador")

indicador
NORMOSTOCK              88548
SOBRESTOCK              27735
SUBSTOCK                25059
DESABASTECIDO           17609
SIN ROTACION            17452
SIN CONSUMO             15304
NO APLICA                8312
NO APLICA DONACIONES     3871
SIN INFORMACION          2784
Name: count, dtype: int64

In [52]:
df_2025_08 = pd.read_excel(
    FILE_2025_08,
    sheet_name="LIMA_METRPOLITANA_2025_08",
    header=3
)

In [53]:
if not PATH_2025_08.exists():
    df_2025_08.to_parquet(
        PATH_2025_08,
        engine="pyarrow",
        index=False,
    )

In [54]:
pt_2025_08 = pd.read_parquet(
    INTERIM_DIR / "pt_2025_08.parquet",
    engine="pyarrow"
)

In [55]:
df_2025_08.shape

(213397, 141)

In [167]:
col_2026_08 = df_2026_08.columns.tolist()
col_2025_08 = df_2025_08.columns.tolist()

filtro_2026_08 = col_2026_08[:27] + col_2026_08[123:]
filtro_2025_08 = col_2025_08[:27] + col_2025_08[123:]

if filtro_2026_08 == filtro_2026_08:
    print("Correcto.")
else: print("Incorrecto.")

Correcto.


In [57]:
# assert df_2026_08.columns.equals(df_2025_08.columns)

In [58]:
# assert df_2026_08.dtypes.equals(df_2025_08.dtypes)

In [59]:
df_2026_08.info(
    verbose=True,
    show_counts=True
)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 206674 entries, 0 to 206673
Data columns (total 141 columns):
 #    Column      Non-Null Count   Dtype         
---   ------      --------------   -----         
 0    codigo_eje  206674 non-null  int64         
 1    nombre_eje  206674 non-null  object        
 2    coddisa     206674 non-null  int64         
 3    nomdisa     206674 non-null  object        
 4    codred      201810 non-null  float64       
 5    red         201810 non-null  object        
 6    codmef      201810 non-null  float64       
 7    uemef       201227 non-null  object        
 8    codigo_pre  206674 non-null  string        
 9    establec    206674 non-null  object        
 10   tipo        206674 non-null  object        
 11   CATEGORIA   194841 non-null  object        
 12   est_act     206674 non-null  object        
 13   ue          31385 non-null   object        
 14   quintil     206674 non-null  int64         
 15   vraem       186843 non-null  obj

In [60]:
# Parte 3:

def count_column_types(df, column):
    return (
        df[column]
        .map(lambda x: type(x).__name__)
        .value_counts()
        .rename_axis("type")
        .rename("frequency")
    )

In [61]:
for column in categorical_columns:
    print(f"\n{'=' * 60}")
    print(f" COLUMN: {column}")
    print(f"{'-' * 60}")
    
    print(
        count_column_types(
            df_2026_08, 
            column
        ).to_string()
    )


 COLUMN: nombre_eje
------------------------------------------------------------
type
str    206674

 COLUMN: nomdisa
------------------------------------------------------------
type
str    206674

 COLUMN: red
------------------------------------------------------------
type
str      201810
float      4864

 COLUMN: uemef
------------------------------------------------------------
type
str      201227
float      5447

 COLUMN: codigo_pre
------------------------------------------------------------
type
str    206674

 COLUMN: establec
------------------------------------------------------------
type
str    206674

 COLUMN: tipo
------------------------------------------------------------
type
str    206674

 COLUMN: CATEGORIA
------------------------------------------------------------
type
str      194841
float     11833

 COLUMN: est_act
------------------------------------------------------------
type
str    206674

 COLUMN: ue
---------------------------------------------------

In [62]:
def count_float_values(df, column):
    return df[
        df[column].apply(lambda x: isinstance(x, float))
    ][column].value_counts()

In [63]:
count_float_values(df_2026_08, "red")

Series([], Name: count, dtype: int64)

In [64]:
df_2026_08["red"] = df_2026_08["red"].astype("string")

In [65]:
get_value_counts(df_2026_08, "red")

red
NO PERTENECE A NINGUNA RED                          186711
LIMA CIUDAD                                           6122
<NA>                                                  4864
LIMA ESTE METROPOLITANA                               3202
SAN JUAN DE LURIGANCHO                                2130
TUPAC AMARU                                           1347
V  RIMAC-SMP-LO                                       1273
RED DE SALUD LIMA NORTE IV                             666
SAN JUAN DE MIRAFLORES - VILLA MARIA DEL TRIUNFO       335
                                                        24
Name: count, dtype: Int64

In [66]:
# Limpieza

In [8]:
df_filtered = df_2026_08.drop(columns=["vraem", "europ", "indig", "mesano"])

In [9]:
df_filtered = df_filtered.drop_duplicates()

In [10]:
df_filtered = df_filtered.dropna(how="all")

In [ ]:
dicc_stip = {
    "INSUMOS": "I",
    "MEDICAMENTO": "M",
    "MEDICAMENTOS": "M",
    "PRODUCTO BIOLÓGICO": "PRODUCTO BIOLOGICO",
    "PRODUCTO DIETÉTICO": "PRODUCTO DIETETICO",
    "NO_DIGESA": "ND"
}

dicc_nmst = {
    "MEDICAMENTOS": "MEDICAMENTO",
    "M": "MEDICAMENTO",
    "INSUMOS": "INSUMO",
    "IINSUMO": "INSUMO",
    "PRODUCTO BIOLÓGICO": "PRODUCTO BIOLOGICO",
    "PRODUCTO DIETÉTICO": "PRODUCTO DIETETICO",
    "INTRUMENTAL ODONTOLOGICO": "INSTRUMENTAL ODONTOLOGICO",
    "GALENICOS": "PRODUCTO GALENICO",
    "ANTISEPTICOS Y DESINFECTANTES": "ANTISEPTICO Y DESINFECTANTE",
    "NO_DIGESA": "NO DIGESA"
}

In [12]:
df_filtered["subtipo"] = df_filtered["subtipo"].replace(dicc_stip)

In [13]:
df_filtered["Nomsubtipo"] = df_filtered["Nomsubtipo"].replace(dicc_nmst)

In [14]:
df_filtered = df_filtered[df_filtered["tipo"] == "HOSPITAL"]

In [15]:
df_filtered.shape

(23330, 137)

In [16]:
df_filtered["uemef"].nunique() # Hospitales diferentes.

18

In [17]:
cols_required = ["codigo_eje", "coddisa", "codred", "codmef", "codigo_pre", "formaf"]
df_filtered = df_filtered.dropna(subset=cols_required)

In [18]:
df_filtered.shape

(23329, 137)

In [21]:
get_value_counts(df_2026_08, "codred")

codred
0.0     186735
21.0      6122
NaN       4864
1.0       3202
2.0       2465
24.0      1347
25.0      1273
26.0       666
Name: count, dtype: int64

In [22]:
for col in df_2026_08.columns:
    missing = df_2026_08[col].isna().sum()

    if missing > 0:
        print(f"{col}: {missing} missing registers.")

codred: 4864 missing registers.
red: 4864 missing registers.
codmef: 4864 missing registers.
uemef: 5447 missing registers.
CATEGORIA: 11833 missing registers.
ue: 175289 missing registers.
vraem: 19831 missing registers.
europ: 19831 missing registers.
indig: 19831 missing registers.
nombre_med: 1 missing registers.
formaf: 3 missing registers.
tipomed: 1 missing registers.
subtipo: 1 missing registers.
Nomsubtipo: 1 missing registers.
petitorio: 1 missing registers.
estrategic: 1 missing registers.
fecha_venc: 33516 missing registers.


In [23]:
df_2026_08["codred"].value_counts(dropna=False)

codred
0.0     186735
21.0      6122
NaN       4864
1.0       3202
2.0       2465
24.0      1347
25.0      1273
26.0       666
Name: count, dtype: int64

In [24]:
for col in df_filtered.columns:
    missing = df_filtered[col].isna().sum()

    if missing > 0:
        print(f"{col}: {missing} missing registers.")

fecha_venc: 4711 missing registers.


In [25]:
def exclude_categories(df, column, categories_to_exclude):
    return df.loc[~df[column].isin(categories_to_exclude)]

In [26]:
exclude_ta = ["I"]
df_filtered = exclude_categories(df_filtered, "est_act", exclude_ta)

exclude_tm = ["I"]
df_filtered = exclude_categories(df_filtered, "tipomed", exclude_tm)

exclude_st = ["POR DEFINIR", "O"]
df_filtered = exclude_categories(df_filtered, "subtipo", exclude_st)

In [27]:
get_value_counts(df_filtered, "tipomed")

tipomed
M    8410
Name: count, dtype: int64

In [28]:
get_value_counts(df_filtered, "subtipo")

subtipo
M       7295
PB       758
NP       104
PG        91
PD        66
AYDE      59
ADX       13
MQ        10
A          7
L          5
NE         2
Name: count, dtype: int64

In [29]:
get_value_counts(df_filtered, "Nomsubtipo")

Nomsubtipo
MEDICAMENTO                                   7305
PRODUCTO BIOLÓGICO                             758
NUTRICION PARENTERAL                           104
PRODUCTO GALENICO                               91
ANTISEPTICO Y DESINFECTANTE                     59
PRODUCTO DIETETICO                              53
POR DEFINIR                                     13
AGENTE DE DIAGNOSTICO                           13
ALIMENTOS, CEREALES, FORMULAS Y NUTRIENTES       7
NO DIGESA                                        5
NUTRICION ENTERAL                                2
Name: count, dtype: int64

In [ ]:
# Salida

In [ ]:
if not Path(INTERIM_DIR / "report.xlsx").exists():
    df_filtered.to_excel(
        INTERIM_DIR / "report.xlsx",
        index = False
    )